<a href="https://colab.research.google.com/github/Liping-LZ/BDAO_ECDA/blob/main/Big%20Data%20Analytics/Analytics_project__PulseLife_case_study.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PulseLife — Data Analytics Project
### WM9A9-15: Big Data, Analytics & Optimisation

---

**The brief:** PulseLife is a UK digital fitness subscription service. Their cancellation rate has nearly doubled over the past year — from 18% to almost 50%. CEO Maya Patel needs to understand who is cancelling, what is driving it, and who is at risk right now.

**Your role:** Data analytics team. You have customer profile data, engagement data, and exit survey comments. Your job is to find the story in the data and give Maya something she can take to her board.

**This notebook follows the analytics lifecycle:**
1. Data Understanding
2. Data Cleaning — Customers
3. Data Cleaning — Engagement
4. Exploratory Data Analysis
5. Machine Learning — Who is at risk? (prediction)

---
> **How to use this notebook:** Read every markdown cell before running the code. Every output has a business interpretation — do not skip it. Your job is not just to run code, it is to understand what the numbers mean for PulseLife.

## Setup — Load libraries

| Library | What it does |
|---|---|
| `pandas` | Works with tables of data |
| `numpy` | Fast number operations |
| `matplotlib` / `seaborn` | Charts and visualisations |
| `sklearn` | Machine learning |


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 12

print('✅ Libraries loaded')

✅ Libraries loaded


### Load the data

We have three files:
- **pulselife_customers.csv** — member profile data
- **pulselife_engagement.csv** — member behaviour and subscription outcome
- **pulselife_comments.csv** — exit survey comments (introduced later)

Upload the first two files to the folder here before running the next cell.

In [ ]:
customers_raw = pd.read_csv('') # Please insert the data path of your data
engagement_raw = pd.read_csv('') # Please insert the data path of your data

print(f'✅ Customers loaded:  {customers_raw.shape[0]:,} rows, {customers_raw.shape[1]} columns')
print(f'✅ Engagement loaded: {engagement_raw.shape[0]:,} rows, {engagement_raw.shape[1]} columns')

---
## 🔍 Section 1 — Data Understanding

Before cleaning or analysing anything, we need to understand what we have.

**Three questions to answer:**
1. What does each column mean in business terms?
2. What is the shape and scale of the data?
3. What quality issues exist that need fixing?

### 1.1 Customers data dictionary

| Column | Business meaning | Type |
|---|---|---|
| `customer_id` | Unique member identifier | string |
| `signup_date` | Date member joined PulseLife | datetime |
| `plan_type` | monthly or annual subscription | string |
| `age_group` | Member age bracket | string |
| `gender` | Member gender | string |
| `city` | Member city | string |
| `signup_source` | How member found PulseLife | string |
| `monthly_fee_gbp` | Monthly fee paid | string |

### 1.2 Engagement data dictionary

| Column | Business meaning | Type |
|---|---|---|
| `customer_id` | Links to customers file | string |
| `logins_per_month` | Average monthly logins | float |
| `classes_attended_per_month` | Average monthly classes attended | float |
| `nutrition_plan_used` | Whether member used the nutrition feature | boolean |
| `support_tickets_raised` | Number of support tickets submitted | integer |
| `payment_failures` | Number of failed payment attempts | integer |
| `months_subscribed` | Total months as a member | float |
| `cancelled` | Whether member cancelled | integer |
| `cancellation_reason` | Why they cancelled — only filled for cancelled members | string |

### 1.3 Translate CNVO Needs into Data Questions

We now have our project scope from the business understanding
step. Before we touch any code, we need to translate each
business question into a specific data question.

This is the most important step in data understanding —
it stops you running analysis that answers the wrong question.

| CNVO Need | Data Question | Where to look |
|---|---|---|
| Why are members cancelling? | Which columns differ most between cancelled = True and cancelled = False? | engagement file — cancelled, cancellation_reason |
| Is plan type affecting cancellation? | What is the cancellation rate for monthly vs annual members? | customers — plan_type, engagement — cancelled |
| Which members are most likely to cancel next? | Can we predict cancelled = True from behavioural features? | engagement — logins, classes, tickets, failures |
| Are monthly members cancelling more? | What is the cancellation rate split by plan_type? | customers — plan_type, engagement — cancelled |
| What are the highest-impact things to fix? | Which features have the strongest relationship with cancellation? | all columns in engagement file |
| What are members saying when they leave? | What words and sentiment patterns appear in comment_text? | comments — comment_text |

> **Every chart and model we build in this notebook answers
> one of these questions. If an analysis does not connect
> to this table — ask whether it is worth doing.**

### 1.4 Data first look & Standard data quality check


Customers data — first look

In [ ]:
print('Shape:', customers_raw.shape)
customers_raw.head()

Run below checks on every dataset before cleaning. They tell you what needs fixing.

In [ ]:
print('=== CUSTOMERS — Data Quality Check ===')
print()
print('1. Data types:')
print(customers_raw.dtypes)
print()
print('2. Missing values:')
print(customers_raw.isnull().sum())
print()
print('3. Duplicates on customer_id:')
print(customers_raw.duplicated(subset=['customer_id']).sum())
print()
print('4. Unique values — categorical columns:')
categorial_columns = ['plan_type','age_group','gender','city','signup_source']
for col in categorial_columns:
    print(f'  {col}: {customers_raw[col].value_counts()}')
print()
print('5. Numeric ranges:')
numeric_columns = ['monthly_fee_gbp']
print(customers_raw[numeric_columns].describe())

> **What to look for:** Notice that `signup_date` is stored as `object` (text), `monthly_fee_gbp` contains £ and $ symbols, and `city` has inconsistent capitalisation. These are the issues we will fix in cleaning.

**Your turn** - can you do the data quality check for the engagement data?

Engagement data — first look

In [ ]:
print(engagement_raw.shape)
engagement_raw.head()

(15000, 9)


,customer_id,logins_per_month,classes_attended_per_month,nutrition_plan_used,support_tickets_raised,payment_failures,months_subscribed,cancelled,cancellation_reason
0,PL000001,11.4,8.0,True,1,0,5.0,1,technical
1,PL000002,0.0,7.9,True,0,0,4.0,0,NaN
2,PL000003,16.3,4.7,True,0,0,11.0,1,competitor
3,PL000004,13.6,8.1,True,1,0,27.0,0,NaN
4,PL000005,3.7,8.3,False,0,0,15.0,1,motivation


Run below checks on every dataset before cleaning. They tell you what needs fixing.

In [ ]:
print('=== ENGAGEMENT — Data Quality Check ===')
print()
print('1. Data types:')
print(engagement_raw.dtypes)
print()
print('2. Missing values:')
print(engagement_raw.isnull().sum())
print()
print('3. Duplicates on customer_id:')
print(engagement_raw.duplicated(subset=['customer_id']).sum())
print()
print('4. Unique values — cancelled column:')
categorial_columns = ['nutrition_plan_used','cancelled','cancellation_reason']
for col in categorial_columns:
    print(f'  {col}: {engagement_raw[col].value_counts()}')
print()
print('5. Numeric ranges:')
numeric_columns = ['logins_per_month','classes_attended_per_month',
                   'support_tickets_raised','months_subscribed']
print(engagement_raw[numeric_columns].describe())

=== ENGAGEMENT — Data Quality Check ===

1. Data types:
customer_id                    object
logins_per_month              float64
classes_attended_per_month    float64
nutrition_plan_used              bool
support_tickets_raised          int64
payment_failures                int64
months_subscribed             float64
cancelled                       int64
cancellation_reason            object
dtype: object

2. Missing values:
customer_id                      0
logins_per_month               433
classes_attended_per_month       0
nutrition_plan_used              0
support_tickets_raised           0
payment_failures                 0
months_subscribed              316
cancelled                        0
cancellation_reason           7796
dtype: int64

3. Duplicates on customer_id:
0

4. Unique values — cancelled column:
  nutrition_plan_used: nutrition_plan_used
False    8438
True     6562
Name: count, dtype: int64
  cancelled: cancelled
0    7796
1    7204
Name: count, dtype: int64
  c

> **Issues found in engagement data:**
> - `logins_per_month` has missing values and outliers (max looks suspicious)
> - `months_subscribed` has missing values
> - `cancelled` stored as string '0'/'1' — needs converting to boolean
> - `cancellation_reason` is missing for active members — this is expected, not an error

---
## 🧹 Section 2 — Data Cleaning: Customers

We will clean the customers file together. Watch each step — you will apply the same process to the engagement file.

**Cleaning order:**
1. Fix data types
2. Standardise text values
3. Handle missing values
4. Validate and document

### 2.1 Fix signup_date — mixed date formats

In [ ]:
# How many rows have the YYYY-MM-DD format?
mixed = customers_raw['signup_date'].str.contains('-').sum()
print(f'Rows with YYYY-MM-DD format: {mixed:,}')
print(f'Rows with other format: {len(customers_raw) - mixed:,}')
print()

In [ ]:
# Fix — format='mixed' handles both automatically
customers = customers_raw.copy()
customers['signup_date'] = pd.to_datetime(
    customers['signup_date'], format='mixed', dayfirst=True
)

print('✅ signup_date fixed')
print(f'Date range: {customers["signup_date"].min().date()} to {customers["signup_date"].max().date()}')
print(f'Data type: {customers["signup_date"].dtype}')

### 2.2 Fix monthly_fee_gbp — mixed currency symbols

In [ ]:
# Check what symbols exist
print('Currency symbols found:')
print(customers['monthly_fee_gbp'].str[0].value_counts())
print()

In [ ]:
# Strip symbol and convert to float
customers['monthly_fee_gbp'] = (
    customers['monthly_fee_gbp']
    .str.replace('£', '', regex=False)
    .str.replace('$', '', regex=False)
    .astype(float)
)

print('✅ monthly_fee_gbp fixed')
print(f'Fee range: £{customers["monthly_fee_gbp"].min()} to £{customers["monthly_fee_gbp"].max()}')
print(f'Unique fees: {sorted(customers["monthly_fee_gbp"].unique())}')

### 2.3 Fix city — inconsistent capitalisation

In [ ]:
print('City values before cleaning:')
print(sorted(customers['city'].unique()))
print()

In [ ]:
# Strip whitespace and apply title case
customers['city'] = customers['city'].str.strip().str.title()

print('City values after cleaning:')
print(sorted(customers['city'].unique()))

### 2.4 Handle missing gender

In [ ]:
# Fill with 'Not specified' — we do not drop because we still want these members in analysis
customers['gender'] = customers['gender'].fillna('Not specified')

print('✅ Missing gender filled with Not specified')
print(customers['gender'].value_counts())

### 2.5 Cleaning summary — customers

| Issue | Fix | Rows affected |
|---|---|---|
| Mixed date formats | Parsed with `format='mixed'` | ~293 rows |
| Mixed currency £/$ | Stripped symbol, converted to float | ~292 rows |
| Inconsistent city case | `.str.title()` | ~453 rows |
| Missing gender | Filled with 'Not specified' | ~283 rows |

> **Document your decisions.** If someone asks why you filled missing gender rather than dropping rows, you can explain: we still want those members in the analysis — their gender is unknown, not irrelevant.

---
## 🧹 Section 3 — Data Cleaning: Engagement

Now apply the same cleaning process to the engagement file. You have already seen what issues exist from the data quality check in Section 1.

**Issues to fix:**
1. `logins_per_month` — missing values and outliers
2. `months_subscribed` — missing values
3. `cancelled` — stored as string '0'/'1', needs converting to boolean

Work through each one below.

### 3.1 Fix logins_per_month — outliers first, then missing values

In [ ]:
# Visualise the histogram of logins_per_month
fig, ax = plt.subplots(figsize=(10, 5))
sns.histplot(data=engagement_raw, x='logins_per_month', ax=ax)
ax.set_title('Logins per Month Distribution', fontweight='bold')
ax.set_xlabel('Logins per Month')

In [ ]:
# Visualise the distribution of logins_per_month to better catch the outliers in boxplot
fig, ax = plt.subplots(figsize=(10, 5))
sns.boxplot(x=engagement_raw['logins_per_month'], ax=ax)
ax.set_title('Logins per Month Distribution', fontweight='bold')
ax.set_xlabel('Logins per Month')
plt.tight_layout()
plt.show()

#### Step 1: Understand the distribution and identify the threshold for capping

Before capping, it's crucial to understand the distribution of the data. The `describe()` method gives us a quick overview, showing the min, max, mean, and quartiles.

We also identify the 99th percentile (P99). This means that 99% of the values in the `logins_per_month` column are below this threshold. Capping at P99 ensures that only the most extreme top 1% of values are adjusted, preserving the general distribution while reducing the influence of very high outliers.

In [ ]:
engagement = engagement_raw.copy()

# It's always good practice to ensure the column is numeric.
# `errors='coerce'` will turn any non-numeric values into NaN, which can then be handled.
engagement['logins_per_month'] = pd.to_numeric(engagement['logins_per_month'], errors='coerce')

print('Initial descriptive statistics for logins_per_month:')
print(engagement['logins_per_month'].describe().round(1))

In [ ]:
# Calculate the 99th percentile of the 'logins_per_month' column.
# This will be our upper limit for capping.
p99 = engagement['logins_per_month'].quantile(0.99)

print(f'99th percentile (P99) for logins_per_month: {p99:.2f}')
print(f'Number of rows where logins_per_month is above P99: {(engagement["logins_per_month"] > p99).sum()}')

#### Step 2: Apply capping to the outliers

The `.clip(upper=p99)` method is used to cap the values. Any value in the `logins_per_month` column that is greater than `p99` will be replaced with `p99`. Values below `p99` remain unchanged.

This method is chosen over simply removing outliers because:
*   It retains all data points, preventing loss of information that might still be relevant.
*   It reduces the skewness and impact of extreme values on statistical measures and models.
*   It's a less aggressive approach than removal, especially when outliers might represent genuine, albeit rare, observations.

In [ ]:
# Apply the capping: all values above p99 are set to p99
engagement['logins_per_month'] = engagement['logins_per_month'].clip(upper=p99)

print('Descriptive statistics for logins_per_month after capping outliers:')
print(engagement['logins_per_month'].describe().round(1))

#### Handle missing values

From previous check, we already know there are missing values (NaNs) in the `logins_per_month` column.

If we drop_na(), then we might lose valuable information in other columns. Given this is a numeric column, filling missing values with the median is a robust strategy:
*   **Median vs. Mean:** The median is less sensitive to outliers than the mean, making it a safer choice when dealing with distributions that might still have some remaining skewness or if the capping wasn't perfectly aggressive.
*   **Preservation of Data:** Like capping, imputation avoids removing rows, ensuring that we retain as much information as possible for subsequent analysis and modeling.

This step ensures that the column is fully populated and ready for use in models.

In [ ]:
print(f'Missing values in logins_per_month before filling: {engagement["logins_per_month"].isna().sum()}')

# Calculate the median of the capped 'logins_per_month' column.
median_logins = engagement['logins_per_month'].median()

# Fill any remaining missing (NaN) values with the calculated median.
engagement['logins_per_month'] = engagement['logins_per_month'].fillna(median_logins)

print(f'Median used for imputation: {median_logins:.2f}')
print(f'Missing values in logins_per_month after filling: {engagement["logins_per_month"].isna().sum()}')
print('\nFinal descriptive statistics for logins_per_month:')
print(engagement['logins_per_month'].describe().round(1))

print('\n✅ logins_per_month cleaning complete.')

### 3.2 Fix months_subscribed — missing values

**Your turn** - can you also handle the missing value for `months_subscribed`

In [ ]:
# Fill with median
median_months = engagement['months_subscribed'].median()
engagement['months_subscribed'] = engagement['months_subscribed'].fillna(median_months).astype(int)

print('✅ months_subscribed fixed')
print(f'Range: {engagement["months_subscribed"].min()} to {engagement["months_subscribed"].max()} months')

✅ months_subscribed fixed
Range: 1 to 30 months


### 3.3 Fix cancelled — convert string to boolean

In [ ]:
# Convert '1'/'0' string to proper boolean
engagement['cancelled'] = engagement['cancelled'].astype(bool)

print('✅ cancelled fixed')
print(engagement['cancelled'].value_counts())
print(f'Data type: {engagement["cancelled"].dtype}')

### 3.4 Merge customers and engagement

In [ ]:
# Join on customer_id
df = customers.merge(engagement, on='customer_id', how='inner')

print(f'✅ Merged dataset: {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'Remaining missing values: {df.isnull().sum()[df.isnull().sum()>0].to_dict()}')
print(f'Duplicates: {df.duplicated(subset=["customer_id"]).sum()}')
print()
print('Overall cancellation rate:', f"{df['cancelled'].mean():.1%}")

> **The data is now clean and merged. One row per member. Ready for analysis.**
> Note: `cancellation_reason` is missing for active members — this is expected. They have not cancelled so there is no reason to record.

---
## 📊 Section 4 — Exploratory Data Analysis

EDA is where we start answering business questions. We do not start with a conclusion — we start with a question and let the data answer it.

> **After every chart ask:** *So what does this mean for PulseLife?*

We will answer five questions:
1. Does plan type predict cancellation?
2. Does engagement predict cancellation?
3. Does support ticket history predict cancellation?
4. When in the membership journey do people cancel?
5. Does acquisition source matter?

### 4.1 Does plan type predict cancellation?

In [ ]:
plan_stats = df.groupby('plan_type').agg(
    members=('customer_id','count'),
    cancel_rate=('cancelled','mean')
).round(3).reset_index()

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Members by plan
axes[0].bar(plan_stats['plan_type'], plan_stats['members'],
            color=['#2196F3','#FF5722'])
axes[0].set_title('Members by Plan Type', fontweight='bold')
axes[0].set_ylabel('Members')

# Cancel rate by plan
bars = axes[1].bar(plan_stats['plan_type'],
                   plan_stats['cancel_rate'] * 100,
                   color=['#2196F3','#FF5722'])
axes[1].set_title('Cancellation Rate by Plan Type', fontweight='bold')
axes[1].set_ylabel('Cancellation Rate (%)')
axes[1].set_ylim(0, 80)
for bar, val in zip(bars, plan_stats['cancel_rate']):
    axes[1].text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + 1, f'{val:.1%}',
                 ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

print(plan_stats.to_string(index=False))

> **Business interpretation:** Monthly members cancel at nearly twice the rate of annual members. This makes intuitive sense — annual members have made a bigger financial commitment. **Recommendation: incentivise members to choose annual plans at signup.**

### 4.2 Does engagement predict cancellation?

In [ ]:
df['login_band'] = pd.cut(df['logins_per_month'],
    bins=[0,5,10,15,100],
    labels=['<5 (low)','5-10','10-15','>15 (high)'], right=False)

df['class_band'] = pd.cut(df['classes_attended_per_month'],
    bins=[0,3,6,10,100],
    labels=['<3 (low)','3-6','6-10','>10 (high)'], right=False)

login_stats = df.groupby('login_band', observed=True)['cancelled'].mean().reset_index()
class_stats = df.groupby('class_band', observed=True)['cancelled'].mean().reset_index()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors_login = ['#FF5722','#FF9800','#FFC107','#4CAF50']
axes[0].bar(login_stats['login_band'],
            login_stats['cancelled'] * 100, color=colors_login)
axes[0].set_title('Cancellation Rate by Monthly Logins', fontweight='bold')
axes[0].set_ylabel('Cancellation Rate (%)')
axes[0].set_xlabel('Logins per Month')

colors_class = ['#FF5722','#FF9800','#FFC107','#4CAF50']
axes[1].bar(class_stats['class_band'],
            class_stats['cancelled'] * 100, color=colors_class)
axes[1].set_title('Cancellation Rate by Classes Attended', fontweight='bold')
axes[1].set_ylabel('Cancellation Rate (%)')
axes[1].set_xlabel('Classes per Month')

plt.tight_layout()
plt.show()

print('Cancellation rate by login band:')
print(login_stats.round(3).to_string(index=False))

> **Business interpretation:** Monthly members cancel at nearly twice the rate of annual members. This makes intuitive sense — annual members have made a bigger financial commitment. **Recommendation: incentivise members to choose annual plans at signup.**

### 4.3 Do support tickets predict cancellation?

In [ ]:
ticket_stats = df.groupby('support_tickets_raised').agg(
    members=('customer_id','count'),
    cancel_rate=('cancelled','mean')
).reset_index()

fig, ax = plt.subplots(figsize=(9, 5))
colors = ['#4CAF50','#FFC107','#FF5722','#B71C1C']
bars = ax.bar(ticket_stats['support_tickets_raised'].astype(str),
              ticket_stats['cancel_rate'] * 100, color=colors)
ax.set_title('Cancellation Rate by Support Tickets Raised', fontweight='bold')
ax.set_xlabel('Number of Support Tickets')
ax.set_ylabel('Cancellation Rate (%)')
ax.set_ylim(0, 80)
for bar, val in zip(bars, ticket_stats['cancel_rate']):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 1, f'{val:.1%}',
            ha='center', fontweight='bold')
ax.axhline(y=df['cancelled'].mean()*100, color='grey',
           linestyle='--', label='Average')
ax.legend()
plt.tight_layout()
plt.show()

print(ticket_stats.round(3).to_string(index=False))

> **Business interpretation:** Monthly members cancel at nearly twice the rate of annual members. This makes intuitive sense — annual members have made a bigger financial commitment. **Recommendation: incentivise members to choose annual plans at signup.**

### 4.4 When in the membership journey do people cancel?

In [ ]:
df['tenure_band'] = pd.cut(df['months_subscribed'],
    bins=[0,3,6,12,100],
    labels=['<3 months','3-6 months','6-12 months','12+ months'], right=False)

tenure_stats = df.groupby('tenure_band', observed=True).agg(
    members=('customer_id','count'),
    cancel_rate=('cancelled','mean')
).reset_index()

fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#FF5722','#FF9800','#FFC107','#4CAF50']
bars = ax.bar(tenure_stats['tenure_band'],
              tenure_stats['cancel_rate'] * 100, color=colors)
ax.set_title('Cancellation Rate by Membership Tenure', fontweight='bold')
ax.set_xlabel('Months as a Member')
ax.set_ylabel('Cancellation Rate (%)')
ax.set_ylim(0, 75)
for bar, val in zip(bars, tenure_stats['cancel_rate']):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 1, f'{val:.1%}',
            ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

print(tenure_stats.round(3).to_string(index=False))

> **Business interpretation:** Monthly members cancel at nearly twice the rate of annual members. This makes intuitive sense — annual members have made a bigger financial commitment. **Recommendation: incentivise members to choose annual plans at signup.**

### 4.5 Does acquisition source matter?

In [ ]:
source_stats = df.groupby('signup_source').agg(
    members=('customer_id','count'),
    cancel_rate=('cancelled','mean')
).sort_values('cancel_rate', ascending=False).reset_index()

fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#FF5722' if r > df['cancelled'].mean() else '#4CAF50'
          for r in source_stats['cancel_rate']]
bars = ax.bar(source_stats['signup_source'],
              source_stats['cancel_rate'] * 100, color=colors)
ax.axhline(y=df['cancelled'].mean()*100, color='grey',
           linestyle='--', label='Average')
ax.set_title('Cancellation Rate by Signup Source', fontweight='bold')
ax.set_ylabel('Cancellation Rate (%)')
ax.legend()
for bar, val in zip(bars, source_stats['cancel_rate']):
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 1, f'{val:.1%}',
            ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

print(source_stats.round(3).to_string(index=False))

> **Business interpretation:** Monthly members cancel at nearly twice the rate of annual members. This makes intuitive sense — annual members have made a bigger financial commitment. **Recommendation: incentivise members to choose annual plans at signup.**

---
## 🤖 Section 5 — Machine Learning: Who is at Risk?

EDA tells us *what patterns exist*. Machine learning tells us *which individual members are most likely to cancel next*.

### How a Random Forest works

Imagine an experienced analyst who has reviewed thousands of member histories. Over time they develop an instinct:

> *'Monthly plan, logs in fewer than 5 times a month, raised two support tickets in month 3 — this person is very likely to cancel.'*

A Random Forest learns exactly this — it builds 100 decision trees from historical data and combines their predictions. Each tree votes, and the majority wins.

**What we are building:** A model that scores every active member with a cancellation probability — so the retention team knows who to contact first.

### 5.1 Prepare the data for modelling

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, ConfusionMatrixDisplay, confusion_matrix
from sklearn.preprocessing import LabelEncoder

# First define what features you are using for training the model
feature_cols = [
    'plan_type', 'age_group', 'gender', 'city', 'signup_source',
    'monthly_fee_gbp', 'logins_per_month', 'classes_attended_per_month',
    'nutrition_plan_used', 'support_tickets_raised', 'payment_failures',
    'months_subscribed'
]

model_data = df[feature_cols + ['cancelled']].copy()

# Encode categorical columns as numbers
categorial_columns = ['plan_type','age_group','gender','city','signup_source']
for col in categorial_columns:
    model_data[col] = LabelEncoder().fit_transform(model_data[col].astype(str))

model_data['nutrition_plan_used'] = model_data['nutrition_plan_used'].astype(int)
model_data['cancelled'] = model_data['cancelled'].astype(int)

X = model_data[feature_cols]
y = model_data['cancelled']

# Split 80% train / 20% test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Training set: {len(X_train):,} members')
print(f'Test set:     {len(X_test):,} members')
print()
print('💡 We train on 80% and test on 20%.')
print('   The test set simulates members the model has never seen before.')

### 5.2 Train the model

In [ ]:
model = RandomForestClassifier(
    n_estimators=100,   # number of trees in the forest
    max_depth=8,        # how many decisions each tree can make
    class_weight='balanced',  # treats cancelled and active members equally — important if our classes are uneven
    random_state=42     # ensures results are reproducible
)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print(f'✅ Model trained')
print(f'Accuracy: {(y_pred == y_test).mean():.1%}')
print()
print(classification_report(y_test, y_pred, target_names=['Active','Cancelled']))



**How to read the results**
`Accuracy`: This is the overall percentage of correct predictions the model made. It means that the model correctly identified whether a member would cancel or not in approximately 64.3% of the cases in the test set.

`Active vs. Cancelled`: These are the two classes our model is trying to predict.

`support`: This is the number of actual instances for each class in the test set. For example, there were 1,559 members who were actually 'Active' (did not cancel) and 1,441 members who actually 'Cancelled' in our test set.

`precision`: This metric tells us, for each class, out of all the instances the model predicted to be of that class, how many were actually correct.


*   **For Active (0.66)**: When the model predicted a member would be Active, it was correct 66% of the time.
*   **For Cancelled (0.62)**: When the model predicted a member would Cancel, it was correct 62% of the time. This means 38% of the members flagged for cancellation might not actually cancel (these are 'false positives').

`recall`: This metric tells us, for each class, out of all the instances that actually belonged to that class, how many did the model correctly identify.


*   **For Active (0.63)**: Of all the members who were actually Active, the model correctly identified 63% of them.
*   **For Cancelled (0.66)**: Of all the members who actually Cancelled, the model correctly identified 66% of them. This is important for retention efforts, as it indicates the model is reasonably good at catching actual churners.

`f1-score`: This is the harmonic mean of precision and recall. It's a good single metric to evaluate models, especially when there might be an uneven class distribution (though here the classes are fairly balanced). A higher F1-score indicates a better balance between precision and recall.

`macro avg`: This is the unweighted average of the precision, recall, and F1-score for both classes. It treats both classes equally.

`weighted avg`: This is the average of the precision, recall, and F1-score for both classes, weighted by the number of instances (support) for each class. Since our classes are relatively balanced, macro avg and weighted avg are similar.



> **How to read accuracy:** 64.3% means the model correctly identifies whether a member will cancel in roughly 2 out of 3 cases. It is not perfect — but it is far better than guessing, and good enough to prioritise who the retention team contacts.

### 5.3 Confusion matrix — what did we get right and wrong?

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(cm, display_labels=['Active','Cancelled'])
disp.plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title('Confusion Matrix — PulseLife Cancellation Model', fontweight='bold')
plt.tight_layout()
plt.show()
print()
print('Reading the matrix:')
print('  Top-left:     Correctly predicted Active')
print('  Top-right:    Predicted Active but actually Cancelled (missed churn)')
print('  Bottom-left:  Predicted Cancelled but actually Active (unnecessary outreach)')
print('  Bottom-right: Correctly predicted Cancelled (our goal)')

### 5.4 What drives cancellation? — Feature importance

In [ ]:
fi = pd.DataFrame({
    'feature': feature_cols,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=True)

fig, ax = plt.subplots(figsize=(10, 7))
colors = ['#FF5722' if f in ['logins_per_month','classes_attended_per_month',
          'plan_type','support_tickets_raised'] else '#90A4AE'
          for f in fi['feature']]
ax.barh(fi['feature'], fi['importance'] * 100, color=colors)
ax.set_title('What Predicts Cancellation?\n(Feature Importance)', fontweight='bold')
ax.set_xlabel('Importance (%)')
plt.tight_layout()
plt.show()

print('Top 5 predictors:')
print(fi.tail(5)[['feature','importance']]
      .sort_values('importance', ascending=False)
      .round(3).to_string(index=False))

### 5.5 Flag at-risk active members

In [ ]:
# Score all members
all_probs = model.predict_proba(X)[:, 1]
df['cancel_probability'] = all_probs

df['risk_tier'] = pd.cut(
    df['cancel_probability'],
    bins=[0, 0.33, 0.66, 1.0],
    labels=['Low Risk', 'Medium Risk', 'High Risk']
)

# Active members only
active = df[df['cancelled'] == False].copy()

print('Risk breakdown — active members:')
print(active['risk_tier'].value_counts())
print()

print('Top 10 highest-risk active members:')
at_risk = active.nlargest(10, 'cancel_probability')[
    ['customer_id','plan_type','city','logins_per_month',
     'support_tickets_raised','months_subscribed','cancel_probability']
]
print(at_risk.round(2).to_string(index=False))

---
## 🎯 Summary — What Have We Found?

| Business Question | Finding | Recommendation |
|---|---|---|
| Who is cancelling? | Monthly plan members at 57% vs annual at 32% | Incentivise annual plans at signup |
| What predicts cancellation? | Low logins, few classes, support tickets | Monitor engagement in first 90 days |
| When do members cancel? | First 3 months is the critical window | Onboarding and early engagement programme |
| Does acquisition matter? | Paid ad members cancel most | Review marketing spend quality |
| Who to contact now? | High Risk tier — ranked at-risk list | Retention team targets top 20% at-risk |
| Why are members leaving? | Technical: angry. Motivation: disengaged. Life change: neutral. | Different response per reason |

### The narrative for Maya's board:
> *'We are losing members because of low engagement in the first 90 days and unresolved technical frustration. We know who is most at risk right now. We know what they need to hear. We have a plan.'*



---
> **This is what the analytics lifecycle looks like in practice.** You started with three raw files and ended with six specific, evidence-based recommendations. That is the job.